In [ ]:
# ==============================================================================
# Task: Mental Health Support Chatbot (Fine-Tuning Assignment)
# Objective: Fine-tune DistilGPT2 on emotional/empathetic dialogue data to
#            provide gentle, supportive responses for wellness conversations.
# ==============================================================================

# ------------------------------------------------------------------------------
# STEP 1: Dependencies Installation
# ------------------------------------------------------------------------------
print("⏳ Installing required libraries (transformers, datasets, accelerate)...")
!pip install -q transformers datasets accelerate

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

# Set device to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")

# ------------------------------------------------------------------------------
# STEP 2: Dataset Loading & Preprocessing
# ------------------------------------------------------------------------------
print("⏳ Loading parquet-formatted 'empathetic-dialogues-cleaned' dataset...")
dataset = load_dataset("chill-bots/empathetic-dialogues-cleaned")

# Initialize Model and Tokenizer
MODEL_NAME = "distilgpt2"
print(f"⏳ Initializing tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    """
    Extracts sentences from the message objects and structures them
    into standard Causal Language Modeling strings.
    """
    texts = []
    for messages in examples["messages"]:
        if len(messages) >= 2:
            user_msg = messages[0]["content"]
            bot_msg = messages[1]["content"]
            texts.append(f"Context: {user_msg}\nResponse: {bot_msg}{tokenizer.eos_token}")
        else:
            texts.append(f"Context: Feeling overwhelmed.\nResponse: I am here to support you.{tokenizer.eos_token}")

    tokenized = tokenizer(texts, truncation=True, max_length=128, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("⏳ Preprocessing dataset splits...")
base_data = dataset["train"]
small_train_dataset = base_data.select(range(0, 1000)).map(preprocess_function, batched=True, remove_columns=base_data.column_names)
small_eval_dataset = base_data.select(range(1000, 1100)).map(preprocess_function, batched=True, remove_columns=base_data.column_names)

# ------------------------------------------------------------------------------
# STEP 3: Model Configuration & Fine-Tuning Setup
# ------------------------------------------------------------------------------
print(f"⏳ Loading base model: {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

training_args = TrainingArguments(
    output_dir="./empathetic_chatbot_results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    data_collator=data_collator
)

print("\n🚀 Launching Fine-Tuning Sequence...")
trainer.train()
print("✅ Fine-tuning completed successfully!")

# ------------------------------------------------------------------------------
# STEP 4: Deployment & Optimized Testing Interface (FIXED REPETITION)
# ------------------------------------------------------------------------------
def generate_support_response(user_input: str) -> str:
    """
    Generates a clean, non-repetitive response using advanced generation parameters.
    """
    model.eval()
    formatted_prompt = f"Context: {user_input}\nResponse:"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            temperature=0.6,          # Lower temperature slightly to reduce erratic tangents
            top_k=40,                 # Limits choices to top plausible words
            top_p=0.85,               # Nucleus sampling value
            repetition_penalty=1.2,   # Penalizes tokens that have already appeared
            no_repeat_ngram_size=2,   # Hard preventer for loops of 2-word patterns
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response_only = decoded.split("Response:")[-1].strip()
    return response_only

# Command-Line Testing Interface Simulation
print("\n" + "="*60 + "\n💬 OPTIMIZED EMOTIONAL SUPPORT CHATBOT INTERFACE\n" + "="*60)

sample_prompts = [
    "I have been feeling really stressed out with my exams lately.",
    "I'm feeling down because things didn't go well today."
]

for prompt in sample_prompts:
    print(f"🔹 User: {prompt}")
    print(f"🔸 Chatbot: {generate_support_response(prompt)}")
    print("-" * 40)